# 🚀 Cocopila Financial Data Agent Pipeline (Kaggle Bootstrap)

Notebook này chứa **toàn bộ mã nguồn Agent Pipeline & Thiết lập môi trường Kaggle** bao gồm:
0. **Kaggle Clone & Workspace Setup**: Clone mã nguồn từ Public Repo vào `/kaggle/working/r2AI_2026`.
1. **Môi trường & Phụ thuộc**: Cài đặt Ollama Linux Binary, Python dependencies (`qdrant-client`, `sentence-transformers`, `rank-bm25`, `langgraph`, `thefuzz`, ...) & pull model `qwen2.5-coder:1.5b`.
2. **System Configuration, Provider & Utilities**: Cấu hình hệ thống, kết nối LLM, JSON repair utility.
3. **Prompts Mẫu (YAML Prompt Templates)**: Query Parser, Code Generator & Reflection Debugging.
4. **Agent State Definition**: Shared State Dictionary dùng trong LangGraph.
5. **Toàn bộ 5 Agent Pipeline Nodes**:
   - **Node 1: Query Parser** (Phân tích câu hỏi tài chính thành JSON cấu trúc & trích xuất khái niệm cốt lõi)
   - **Node 2: Data Discovery** (Tìm kiếm bảng dữ liệu phù hợp với Search Engine & DataRegistry)
   - **Node 3: Schema Mapper** (Ánh xạ tiêu chí phụ sang tên cột thực tế trong CSV)
   - **Node 4: Code Generator & Reflection** (Sinh mã Python/Pandas trích xuất/tính toán/so sánh & tự động sửa lỗi)
   - **Node 5: AST Sandbox & Executor** (Thực thi mã Python an toàn trong Sandbox AST)
6. **Workflow StateGraph & Conditional Edge Routing**: Khởi tạo LangGraph app với vòng lặp Reflection Loop.
7. **Kiểm thử trực tiếp trên 10 câu hỏi ngẫu nhiên (seed=42)**

## 🛠️ Section 0: Kaggle Workspace Setup & Code Cloning

In [ ]:
# 0. Clone mã nguồn dự án vào thư mục /kaggle/working/r2AI_2026
import os
import sys
import shutil
import subprocess
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "r2AI_2026"
REPO_URL = "https://github.com/Djuybu/r2AI_2026.git"

# Kiểm tra Token nếu repo là Private (Lấy từ Kaggle Secrets hoặc biến môi trường)
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
except Exception:
    pass

if GITHUB_TOKEN and "github.com" in REPO_URL:
    auth_repo_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
else:
    auth_repo_url = REPO_URL

if os.path.exists("/kaggle/working"):
    print("🚀 Phát hiện môi trường Kaggle! Đang chuẩn bị thư mục làm việc...")
    if not REPO_DIR.exists():
        print(f"📥 Đang clone repository từ {REPO_URL} vào {REPO_DIR}...")
        res = subprocess.run(["git", "clone", auth_repo_url, str(REPO_DIR)], capture_output=True, text=True)
        if res.returncode != 0:
            print(f"⚠️ Lỗi Git Clone: {res.stderr.strip()}")
            # Dò tìm mã nguồn trong /kaggle/input làm phương án dự phòng
            dataset_candidates = list(Path("/kaggle/input").glob("**/r2AI_2026")) if Path("/kaggle/input").exists() else []
            if dataset_candidates:
                src_path = dataset_candidates[0]
                print(f"📦 Tìm thấy mã nguồn trong Kaggle Input Dataset: {src_path}. Đang sao chép sang {REPO_DIR}...")
                shutil.copytree(src_path, REPO_DIR, dirs_exist_ok=True)

    if REPO_DIR.exists():
        os.chdir(str(REPO_DIR))
        if str(REPO_DIR) not in sys.path:
            sys.path.insert(0, str(REPO_DIR))
        print(f"✅ Đã chuyển thư mục làm việc: {os.getcwd()}")
    else:
        print(f"⚠️ Không tìm thấy thư mục {REPO_DIR}. Tiếp tục với thư mục làm việc mặc định: {os.getcwd()}")
else:
    print(f"💻 Đang chạy trên môi trường Local: {os.getcwd()}")

In [ ]:
# Cài đặt các công cụ hệ thống hỗ trợ
!sudo apt-get update -y
!sudo apt install lshw -y
!sudo apt-get install zstd -y


## 📥 Section 0.1: Installing Ollama CLI & Dependencies on Kaggle

In [ ]:
# 0.1 Cài đặt Ollama CLI trên Linux kernel của Kaggle (nếu chưa có)
print("📥 Đang kiểm tra / cài đặt Ollama CLI trên Kaggle Linux...")
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 1. Cài đặt đầy đủ các gói phụ thuộc dự án (Bao gồm LangGraph, Qdrant Client, Sentence Transformers, BM25, ...)
print("📥 Đang cài đặt Python dependencies cho Agent Pipeline & RAG Search Engine...")
!pip install -q \
    langgraph>=0.2.0 \
    langchain-core>=0.3.0 \
    langchain-openai>=0.2.0 \
    pyyaml>=6.0 \
    json-repair>=0.30.0 \
    openpyxl>=3.1.0 \
    tabulate>=0.9.0 \
    thefuzz>=0.22.0 \
    qdrant-client \
    sentence-transformers \
    rank-bm25

In [ ]:
# 2. Khởi động Ollama Server chạy nền & Tải model
import subprocess
import time
import requests
import os

print("🚀 Đang khởi động Ollama Server...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("⏳ Chờ Ollama Server khởi động...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/")
        if r.status_code == 200:
            print("✅ Ollama Server đã sẵn sàng tại port 11434!")
            break
    except Exception:
        time.sleep(1)
else:
    print("❌ Lỗi: Ollama Server không thể khởi động.")

MODEL_NAME = "qwen2.5-coder:1.5b"
print(f"📥 Đang tải mô hình {MODEL_NAME} từ Ollama registry...")
subprocess.run(["ollama", "pull", MODEL_NAME])
print(f"✅ Đã tải thành công mô hình {MODEL_NAME}!")

os.environ["MODEL_NAME"] = MODEL_NAME
os.environ["LLM_API_BASE"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"] = "ollama"

## ⚙️ Section 1: System Configuration, LLM Provider & Utilities

In [ ]:
import os
import json
import logging
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, List, Optional, Literal, TypedDict
from json_repair import repair_json
from langchain_openai import ChatOpenAI
from langchain_core.language_models.chat_models import BaseChatModel

def is_kaggle_environment() -> bool:
    """Check if execution environment is Kaggle."""
    return os.path.exists("/kaggle/working")

@dataclass
class Config:
    """System configuration parameters."""
    MODEL_NAME: str = os.getenv("MODEL_NAME", "qwen2.5-coder:1.5b")
    LLM_API_BASE: str = os.getenv("LLM_API_BASE", "http://localhost:11434/v1")
    LLM_API_KEY: str = os.getenv("LLM_API_KEY", "ollama")
    TEMPERATURE: float = float(os.getenv("TEMPERATURE", "0.0"))
    MAX_TOKENS: int = int(os.getenv("MAX_TOKENS", "1024"))
    BASE_DIR: Path = Path("/kaggle/working/r2AI_2026/pipeline") if is_kaggle_environment() else Path.cwd()
    DATA_DIR: Path = Path(os.getenv("DATA_DIR", "/kaggle/working/r2AI_2026/pipeline/data"))
    MAX_RETRIES: int = int(os.getenv("MAX_RETRIES", "3"))

config = Config()

def get_llm(
    cfg: Optional[Config] = None,
    temperature: Optional[float] = None,
    max_tokens: Optional[int] = None,
    timeout: Optional[int] = 30,
    **kwargs,
) -> BaseChatModel:
    """Instantiate ChatOpenAI connected to local LLM endpoint safely supporting timeout & kwargs."""
    cfg = cfg or config
    temp = temperature if temperature is not None else cfg.TEMPERATURE
    tokens = max_tokens if max_tokens is not None else cfg.MAX_TOKENS
    llm_kwargs = {
        "model": cfg.MODEL_NAME,
        "openai_api_base": cfg.LLM_API_BASE,
        "openai_api_key": cfg.LLM_API_KEY,
        "temperature": temp,
        "max_tokens": tokens,
        "streaming": False,
    }
    if timeout is not None:
        llm_kwargs["request_timeout"] = timeout
    return ChatOpenAI(**llm_kwargs)

def safe_parse_json(content: str) -> Dict[str, Any]:
    """Parse JSON string with automatic repair fallback."""
    if not content or not content.strip():
        return {}
    cleaned = content.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    try:
        repaired = repair_json(cleaned)
        return json.loads(repaired)
    except Exception as exc:
        print(f"⚠️ Safe JSON parse failed: {exc}")
        return {}

print("✅ System configuration & LLM Provider utilities loaded!")


## 📝 Section 2: Prompts Mẫu (Prompt Templates)
Định nghĩa các Prompt mẫu cho Query Parser, Code Generator và Reflection Debugging Loop.

In [ ]:
PROMPT_QUERY_PARSER = {
    "system_prompt": """You MUST output ONLY a valid JSON object with the exact keys: {"ticker": "string", "year": "string", "metric": "string"}. You must extract the exact financial metric string. You MUST map the company name to the correct ticker using the provided code stock mapping. Do NOT wrap the JSON in markdown code blocks, just output the raw JSON.

CRITICAL INSTRUCTION: The JSON examples provided above are STRICTLY for formatting demonstration. DO NOT copy the values (ticker, year, metric) from the examples. You MUST read the actual [USER_QUERY] provided and dynamically extract the REAL ticker, REAL year, and REAL metric requested by the user.

## DETAILED INSTRUCTIONS:
1. "ticker": Map the target company or bank name in the query to its exact 3-5 letter uppercase ticker symbol using the stock mapping (code_stock.csv). (e.g., "Vietjet" -> "VJC", "Ngân hàng TMCP Sài Gòn Thương Tín" -> "STB", "FPT" -> "FPT", "Tập đoàn Vingroup" -> "VIC"). If no company is mentioned or if the company is unknown, output an empty string "". Do NOT copy tickers from examples!

2. "year": Extract the target year or list of years from the query as a string (e.g., "2023" or "2021, 2022, 2023"). Never output null or None for the year if a year appears in the query!

3. "metric": Extract the exact financial metric string verbatim from the query (e.g., "Lãi tiền gửi", "Chi phí khác", "Doanh thu thuần", "Lãi vay phải trả"). Do NOT copy metrics from examples!

OUTPUT FORMAT: Output ONLY raw valid JSON matching {"ticker": "string", "year": "string", "metric": "string"}.""",
    "user_prompt_template": """Câu hỏi: {user_query}

Ví dụ mẫu:
{few_shot_examples}

Hãy phân tích câu hỏi trên và trả về JSON:""",
    "few_shot_examples": [
        {
            "user_query": "Lãi tiền gửi năm 2021 của Vietjet (VJC) là bao nhiêu triệu đồng?",
            "parsed_output": '{"ticker": "VJC", "year": "2021", "metric": "Lãi tiền gửi"}'
        },
        {
            "user_query": "Chi phí lương và các khoản khác theo lương của công ty mẹ CTCP Chứng khoán FPT trong năm 2021 là bao nhiêu tỷ đồng?",
            "parsed_output": '{"ticker": "FPT", "year": "2021", "metric": "Chi phí lương và các khoản khác theo lương"}'
        },
        {
            "user_query": "Lãi vay phải trả của CTCP Tập đoàn Đức Long Gia Lai (DLG) cuối năm 2023 là bao nhiêu triệu đồng?",
            "parsed_output": '{"ticker": "DLG", "year": "2023", "metric": "Lãi vay phải trả"}'
        },
        {
            "user_query": "Doanh thu hoạt động tài chính năm 2020 của Ngân hàng TMCP Sài Gòn Thương Tín (STB)",
            "parsed_output": '{"ticker": "STB", "year": "2020", "metric": "Doanh thu hoạt động tài chính"}'
        },
        {
            "user_query": "Vốn chủ sở hữu của FIT là bao nhiêu tỷ đồng vào ngày 31/12/2015?",
            "parsed_output": '{"ticker": "FIT", "year": "2015", "metric": "Vốn chủ sở hữu"}'
        },
        {
            "user_query": "Tốc độ tăng trưởng % tổng tiền và các khoản tương đương tiền từ năm 2019 đến năm 2021 của VIC",
            "parsed_output": '{"ticker": "VIC", "year": "2019, 2020, 2021", "metric": "tổng tiền và các khoản tương đương tiền"}'
        }
    ]
}

PROMPT_CODE_GENERATOR = {
    "system_prompt": """Bạn là một chuyên gia lập trình Python & Pandas Data Analysis.
Nhiệm vụ của bạn là sinh ra đoạn mã Python để truy vấn dữ liệu từ bảng báo cáo tài chính.

CRITICAL RULE: DO NOT EVER hardcode or append result = 0.0 or any fallback value at the end of the script to bypass errors. If the filtered dataframe is empty, the script MUST end by raising a ValueError("Metric not found in table"). You MUST let the script crash if the exact metric is not found!
CRITICAL OUTPUT RULE: Output ONLY executable Python code. NO markdown formatting. NO ```python tags. NO conversational text or explanations. Start directly with 'import pandas'.

QUY TẮC BẮT BUỘC (TUYỆT ĐỐI TUÂN THỦ):
1. Dữ liệu nằm trong file CSV. Đọc file bằng `pd.read_csv(file_path)`.
2. Cấu trúc bảng: Cột nhãn (label_column) chứa tên chỉ tiêu. Cột giá trị (value_column) chứa số liệu.
3. BẮT BUỘC dùng `regex=False` khi sử dụng `str.contains` để tránh lỗi regex với các chỉ tiêu có dấu ngoặc tròn (ví dụ: `Lợi nhuận sau thuế (60 = 50 - 51)`).
4. KHÔNG BAO GIỜ được gọi `.iloc[0]` hoặc `.values[0]` trực tiếp trên DataFrame sau khi lọc mà chưa kiểm tra DataFrame có rỗng hay không!
5. BẮT BUỘC sử dụng mẫu code an toàn (Safe Pattern) sau đây:
   filtered_df = df[df['{label_col}'].astype(str).str.contains(..., case=False, na=False, regex=False)]
   if not filtered_df.empty:
       result = clean_val(filtered_df['{value_col}'].iloc[0])
   else:
       raise ValueError("Metric not found in table")
6. XỬ LÝ KHI KHÔNG TÌM THẤY DỮ LIỆU / DỮ LIỆU RỖNG:
   TUYỆT ĐỐI KHÔNG gán `result = 0.0` hay trả về 0.0 ở cuối script để bypass lỗi.
   BẮT BUỘC phải `raise ValueError("Metric not found in table")` để kích hoạt vòng lặp tự sửa lỗi (Reflection Loop).
7. Luôn định nghĩa hàm clean_val(val) như sau (hàm sẽ raise ValueError nếu giá trị rỗng hoặc không hợp lệ):
   def clean_val(val):
       if pd.isna(val) or str(val).strip() in ['-', '', 'nan', 'NaN', 'None', 'n/a', '—']:
           raise ValueError("Metric not found in table")
       val_str = str(val).strip()
       neg = False
       if val_str.startswith('(') and val_str.endswith(')'):
           neg = True
           val_str = val_str[1:-1].strip()
       val_str = val_str.replace(',', '')
       if '.' in val_str:
           parts = val_str.split('.')
           if len(parts) > 2 or (len(parts) == 2 and len(parts[1]) == 3):
               val_str = val_str.replace('.', '')
       try:
           res = float(val_str)
           return -res if neg else res
       except Exception:
           raise ValueError("Metric not found in table")
8. Kết quả cuối cùng BẮT BUỘC gán vào biến `result`.
9. CHỈ trả về code Python thuần túy. KHÔNG giải thích, KHÔNG bọc markdown.
""",
    "goal_descriptions": {
        "trich_xuat": "TRÍCH XUẤT giá trị cụ thể",
        "tinh_tong": "TÍNH TỔNG (tìm dòng Tổng/Cộng trước, nếu không có thì cộng các dòng con)",
        "so_sanh": "SO SÁNH giá trị giữa nhiều năm/công ty"
    },
    "goal_instructions": {
        "trich_xuat": """HƯỚNG DẪN CỤ THỂ (TRÍCH XUẤT):
1. Đọc file CSV bằng pd.read_csv(file_path).
2. Truy vấn hàng ở cột '{label_col}' với `str.contains(..., case=False, na=False, regex=False)`.
3. Dùng kiểm tra `if not filtered_df.empty:` để lấy giá trị cột '{value_col}' qua `clean_val(filtered_df['{value_col}'].iloc[0])`. Nếu empty → raise ValueError("Metric not found in table").
4. Gán vào biến result.""",
        "tinh_tong": """HƯỚNG DẪN CỤ THỂ (TÍNH TỔNG):
1. Đọc file CSV bằng pd.read_csv(file_path).
2. Tìm dòng có chứa 'Tổng' hoặc 'Cộng' hoặc tên chỉ tiêu '{noi_dung}' ở cột '{label_col}' bằng `str.contains(..., regex=False)`.
3. Kiểm tra `if not filtered_df.empty:` lấy giá trị qua `clean_val(...)`. Nếu không tìm thấy, cộng các dòng con liên quan. Nếu vẫn rỗng → raise ValueError("Metric not found in table").
4. Gán kết quả vào result.""",
        "so_sanh": """HƯỚNG DẪN CỤ THỂ (SO SÁNH NĂM / TÍNH TỐC ĐỘ TĂNG TRƯỞNG):
1. Đọc từng file CSV cho từng năm (ví dụ file_path_2019, file_path_2020, file_path_2021...).
2. Truy vấn hàng ở cột '{label_col}' với `str.contains(..., regex=False)` và kiểm tra `if not filtered_df.empty:` cho từng năm. Nếu rỗng → raise ValueError("Metric not found in table").
3. Lấy giá trị cột '{value_col}' qua clean_val().
4. Tính tốc độ tăng trưởng phần trăm (%) giữa năm đầu và năm cuối: `growth_rate = ((val_last - val_first) / val_first) * 100`.
5. Gán kết quả vào `result`."""
    },
    "user_prompt_template": """Yêu cầu người dùng: {user_query}

MỤC TIÊU: {muc_tieu_desc}
NỘI DUNG cần tìm (ở cột label): '{noi_dung}'
Công ty: {ten_cong_ty}
Năm: {so_nam}
Tiêu chí phụ: {tieu_chi_phu}

DỮ LIỆU CÓ SẴN:
{files_context}

CỘT QUAN TRỌNG:
- Cột nhãn (chứa tên chỉ tiêu): '{label_col}'
- Cột giá trị: '{value_col}'

BIẾN ĐƯỜNG DẪN FILE:
{paths_str}

{goal_instruction}

🚨 BẮT BUỘC KHÔNG ĐƯỢC VI PHẠM:
1. CRITICAL RULE: DO NOT EVER hardcode or append result = 0.0 or any fallback value at the end of the script to bypass errors. If the filtered dataframe is empty, the script MUST end by raising a ValueError("Metric not found in table").
2. CRITICAL OUTPUT RULE: Output ONLY executable Python code. NO markdown formatting. NO ```python tags. NO conversational text or explanations. Start directly with 'import pandas'.
3. Định nghĩa clean_val(val) (raise ValueError("Metric not found in table") nếu rỗng/invalid).
4. Đọc file bằng pd.read_csv(file_path...).
5. Dùng mẫu safe pattern với `regex=False`: `filtered_df = df[df['{label_col}'].astype(str).str.contains(..., case=False, na=False, regex=False)]`
6. BẮT BUỘC dùng `if not filtered_df.empty:` trước khi lấy `.iloc[0]`. Nếu rỗng, BẮT BUỘC `raise ValueError("Metric not found in table")`. KHÔNG ĐƯỢC gán result = 0.0!
7. KHÔNG ĐƯỢC filter theo `df['Ma_Doanh_Nghiep'] == ...` vì dữ liệu đã đúng công ty.
8. CHỈ ĐƯỢC SỬ DỤNG CÁC BIẾN ĐƯỜNG DẪN FILE ĐÃ ĐƯỢC ĐỊNH NGHĨA Ở TRÊN:
{paths_str}
9. Kết quả cuối cùng BẮT BUỘC lưu vào biến `result`.""",
    "few_shot_examples": [
        {
            "user_query": "Doanh thu thuần năm 2023 của FPT",
            "file_path": "data/FPT_2023.csv",
            "column_mapping": '{"label_column": "CHỈ TIÊU", "value_column": "Năm nay"}',
            "generated_code": """import pandas as pd

def clean_val(val):
    if pd.isna(val) or str(val).strip() in ['-', '', 'nan', 'NaN', 'None', 'n/a', '—']:
        raise ValueError("Metric not found in table")
    val_str = str(val).strip().replace('.', '')
    if '(' in val_str and ')' in val_str:
        val_str = '-' + val_str.replace('(', '').replace(')', '')
    try:
        return float(val_str)
    except Exception:
        raise ValueError("Metric not found in table")

df = pd.read_csv(file_path)
filtered_df = df[df['CHỈ TIÊU'].astype(str).str.contains('Doanh thu thuần', case=False, na=False, regex=False)]
if not filtered_df.empty:
    result = clean_val(filtered_df['Năm nay'].iloc[0])
else:
    raise ValueError("Metric not found in table")"""
        }
    ]
}

PROMPT_REFLECTION = {
    "system_prompt": """Đoạn mã Pandas trước đó đã gặp lỗi khi thực thi trong Sandbox môi trường.
Nhiệm vụ của bạn là phân tích lỗi Traceback, sửa lại đoạn mã Python/Pandas để đảm bảo không bị lỗi và giải quyết chính xác yêu cầu của người dùng.

CRITICAL REFLECTION RULE:
- If your previous attempt failed because the row was not found, DO NOT write the exact same string matching code again. You must try matching a different variation of the row name, or use partial string matching (str.contains).
- CRITICAL OUTPUT RULE: Output ONLY executable Python code. NO markdown formatting. NO ```python tags. NO conversational text or explanations. Start directly with 'import pandas'.

CRITICAL RULE: DO NOT EVER hardcode or append result = 0.0 or any fallback value at the end of the script to bypass errors. If the filtered dataframe is empty, the script MUST end by raising a ValueError("Metric not found in table"). You MUST let the script crash if the exact metric is not found!

QUY TẮC SỬA LỖI BẮT BUỘC:
1. Phân tích nguyên nhân gây lỗi dựa vào error_traceback.
2. BẮT BUỘC sử dụng Safe Pattern kèm `regex=False`:
   filtered_df = df[df['{label_col}'].astype(str).str.contains(..., case=False, na=False, regex=False)]
   if not filtered_df.empty:
       result = clean_val(filtered_df['{value_col}'].iloc[0])
   else:
       raise ValueError("Metric not found in table")
3. TUYỆT ĐỐI KHÔNG được gọi `.iloc[0]` hoặc `.values[0]` trực tiếp trên DataFrame sau khi lọc mà chưa kiểm tra `if not filtered_df.empty:`.
4. BẮT BUỘC dùng `regex=False` khi gọi `str.contains` để tránh lỗi Regex khi chuỗi chỉ tiêu chứa dấu ngoặc đơn (ví dụ: `(60 = 50 - 51)`).
5. BẮT BUỘC định nghĩa clean_val(val) như sau (hàm sẽ raise ValueError nếu giá trị rỗng/không hợp lệ):
   def clean_val(val):
       if pd.isna(val) or str(val).strip() in ['-', '', 'nan', 'NaN', 'None', 'n/a', '—']:
           raise ValueError("Metric not found in table")
       val_str = str(val).strip()
       neg = False
       if val_str.startswith('(') and val_str.endswith(')'):
           neg = True
           val_str = val_str[1:-1].strip()
       val_str = val_str.replace(',', '')
       if '.' in val_str:
           parts = val_str.split('.')
           if len(parts) > 2 or (len(parts) == 2 and len(parts[1]) == 3):
               val_str = val_str.replace('.', '')
       try:
           res = float(val_str)
           return -res if neg else res
       except Exception:
           raise ValueError("Metric not found in table")
6. XỬ LÝ KHÔNG TÌM THẤY DỮ LIỆU:
   Nếu lọc theo từ khóa dài bị rỗng, hãy thử lọc theo từ khóa ngắn hơn trong `sample_labels_str`. NẾU VẪN RỖNG, BẮT BUỘC `raise ValueError("Metric not found in table")`. TUYỆT ĐỐI KHÔNG gán result = 0.0 hoặc append result = 0.0 ở cuối file!
7. Kết quả cuối cùng BẮT BUỘC lưu vào biến `result`.
8. CHỈ trả về khối mã Python thuần túy. KHÔNG giải thích, KHÔNG bọc markdown.
""",
    "user_prompt_template": """Yêu cầu người dùng: {user_query}

MỤC TIÊU: {muc_tieu}
NỘI DUNG: '{noi_dung}'
DỮ LIỆU:
{files_context}

CỘT: label='{label_col}', value='{value_col}'

BIẾN ĐƯỜNG DẪN:
{paths_str}

{sample_labels_str}

Mã Python bị lỗi trước đó:
```python
{previous_code}
```

Traceback Lỗi:
{error_traceback}

HƯỚNG DẪN SỬA LỖI:
1. If your previous attempt failed because the row was not found, DO NOT write the exact same string matching code again. You must try matching a different variation of the row name, or use partial string matching (str.contains).
2. CRITICAL OUTPUT RULE: Output ONLY executable Python code. NO markdown formatting. NO ```python tags. NO conversational text or explanations. Start directly with 'import pandas'.
3. Dùng mẫu safe pattern với `regex=False`: `filtered_df = df[df['{label_col}'].astype(str).str.contains(..., case=False, na=False, regex=False)]`.
4. BẮT BUỘC dùng `if not filtered_df.empty:` trước khi truy cập `.iloc[0]`. Nếu rỗng, raise ValueError("Metric not found in table").
5. Đảm bảo kết quả cuối cùng BẮT BUỘC lưu vào biến `result`."""
}

print("✅ Prompts templates loaded!")


## 🧠 Section 3: Agent State Definition

In [ ]:
class AgentState(TypedDict, total=False):
    """Shared state dictionary passed across LangGraph nodes."""
    user_query: str
    parsed_query: Dict[str, Any]
    discovered_tables: List[Dict[str, Any]]
    column_mapping: Dict[str, str]
    generated_code: str
    execution_result: Any
    error_traceback: Optional[str]
    retry_count: int
    status: Literal["pending", "success", "error"]
    error_message: Optional[str]
    node_latencies: Dict[str, float]

print("✅ AgentState TypedDict defined!")


## 🧩 Section 4: Agent Pipeline Nodes

### Node 1: Query Parser Node
Trích xuất `ten_cong_ty`, `so_nam`, `noi_dung`, `thao_tac`, `tieu_chi_phu` từ câu hỏi người dùng.

In [ ]:
def parse_query_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()
    user_query = state.get("user_query", "").strip()

    if not user_query:
        return {
            **state,
            "status": "error",
            "error_message": "User query is empty.",
            "parsed_query": {},
        }

    try:
        prompt_data = PROMPT_QUERY_PARSER
        sys_prompt = prompt_data.get("system_prompt", "")
        few_shot = json.dumps(prompt_data.get("few_shot_examples", []), ensure_ascii=False, indent=2)
        
        user_content = prompt_data.get("user_prompt_template", "{user_query}").format(
            user_query=user_query,
            few_shot_examples=few_shot
        )

        # Call LLM with slight temperature=0.1 to avoid deterministic overfitting/copy-pasting examples
        llm = get_llm(cfg, temperature=0.1)
        response = llm.invoke([
            SystemMessage(content=sys_prompt),
            HumanMessage(content=user_content)
        ])
        raw_output = response.content if hasattr(response, "content") else str(response)
        parsed = safe_parse_json(raw_output)

        # Map strict schema keys {"ticker", "year", "metric"} to state schema keys
        ticker_val = parsed.get("ticker") or parsed.get("ten_cong_ty") or ""
        metric_val = parsed.get("metric") or parsed.get("noi_dung") or ""
        year_val = parsed.get("year") if "year" in parsed else parsed.get("so_nam")

        # Fallback year extraction from query if year is None or empty
        if not year_val or year_val is None or str(year_val).strip() in ["None", "null", ""]:
            year_val = re.findall(r"\b(20\d{2})\b", user_query)

        # Ensure so_nam is a list of strings
        if isinstance(year_val, str):
            so_nam_list = [y.strip() for y in year_val.replace(",", " ").split() if y.strip().isdigit()]
            if not so_nam_list:
                so_nam_list = re.findall(r"\b(20\d{2})\b", user_query)
        elif isinstance(year_val, (int, float)):
            so_nam_list = [str(int(year_val))]
        elif isinstance(year_val, list):
            so_nam_list = [str(y).strip() for y in year_val if str(y).strip().isdigit()]
        else:
            so_nam_list = re.findall(r"\b(20\d{2})\b", user_query)

        # Sync keys
        parsed["ticker"] = ticker_val
        parsed["ten_cong_ty"] = _normalize_company_name(ticker_val, user_query)
        parsed["year"] = ", ".join(so_nam_list) if so_nam_list else ""
        parsed["so_nam"] = so_nam_list
        parsed["metric"] = metric_val
        parsed["noi_dung"] = metric_val

        thao_tac = parsed.get("thao_tac") or parsed.get("muc_tieu") or ("so_sanh" if len(so_nam_list) > 1 or "so sánh" in user_query.lower() or "tăng trưởng" in user_query.lower() else "trich_xuat")
        if thao_tac not in ["trich_xuat", "so_sanh"]:
            thao_tac = "trich_xuat"
        parsed["thao_tac"] = thao_tac
        parsed["muc_tieu"] = thao_tac

        raw_noi_dung = parsed.get("noi_dung", "")
        if thao_tac == "so_sanh":
            parsed["noi_dung"] = _clean_financial_content(raw_noi_dung) or raw_noi_dung
            parsed["metric"] = parsed["noi_dung"]

        print(f"📍 Node: [QUERY_PARSER] (Thời gian chạy: {time.time() - start_time:.3f}s)")
        print(f"   Công ty / Ticker: '{parsed.get('ten_cong_ty')}' (gốc: '{ticker_val}')")
        print(f"   Số năm / Year: {parsed.get('so_nam')}")
        print(f"   Nội dung / Metric: '{parsed.get('noi_dung')}'")
        print(f"   Thao tác: {parsed.get('thao_tac')}")

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["query_parser"] = round(latency, 3)

        return {
            **state,
            "parsed_query": parsed,
            "status": "pending",
            "node_latencies": node_latencies,
        }
    except Exception as exc:
        print(f"⚠️ [Query Parser] LLM không phản hồi ({exc}). Sử dụng Fallback Parser...")
        parsed = _fallback_parse_query(user_query)
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["query_parser"] = round(latency, 3)
        return {
            **state,
            "parsed_query": parsed,
            "status": "pending",
            "node_latencies": node_latencies,
        }

# Alias for backwards compatibility
query_parser_node = parse_query_node


### Node 2: Data Discovery Node
Dùng `Search Engine` tra cứu bảng dữ liệu tương ứng theo công ty, năm, nội dung.

In [ ]:
def _resolve_csv_path(csv_path_str: str, cfg: Config) -> Optional[Path]:
    if not csv_path_str:
        return None
    p_str = csv_path_str.replace("\\", "/")
    direct = Path(p_str).resolve()
    if direct.exists():
        return direct
    idx_fin = p_str.find("ViFinQA")
    if idx_fin != -1:
        relative_part = p_str[idx_fin:]
        repo_root = Path("/kaggle/working/r2AI_2026").resolve()
        candidate1 = (repo_root / relative_part).resolve()
        if candidate1.exists():
            return candidate1
        candidate2 = (repo_root / "rag_module" / relative_part).resolve()
        if candidate2.exists():
            return candidate2
        candidate3 = Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data") / relative_part
        if candidate3.exists():
            return candidate3
    return None

def _log_candidates(results: List[Dict[str, Any]], year_label: str = "") -> None:
    prefix = f" (Năm {year_label})" if year_label else ""
    print(f"   📋 Danh sách {len(results)} bảng ứng viên Top-K từ Search Engine{prefix}:")
    for idx, item in enumerate(results, 1):
        p_str = item.get("csv_path", "")
        file_name = Path(p_str).name if p_str else "N/A"
        ten_bang = item.get("Ten_Bang", "N/A")
        rrf = item.get("rrf_score", 0.0)
        sample = str(item.get("matched_sample", "N/A"))
        if len(sample) > 40:
            sample = sample[:37] + "..."
        print(f"      #{idx} RRF: {rrf:.6f} | File: {file_name} | Hàng khớp: '{sample}' | Tên bảng: {ten_bang}")

def clean_query_content(noi_dung_input: str, ticker: str = "", so_nam: list = None) -> str:
    if not noi_dung_input:
        return ""
    text = str(noi_dung_input)
    text = re.sub(r"\([A-Za-z]{2,5}\)", "", text)
    text = re.sub(r"\b20\d{2}\b", "", text)
    if ticker:
        text = re.sub(r"\b" + re.escape(ticker) + r"\b", "", text, flags=re.IGNORECASE)

    patterns = [
        r"là bao nhiêu.*", r"bao nhiêu.*", r"của công ty mẹ.*", r"của ngân hàng.*",
        r"của ctcp.*", r"của tập đoàn.*", r"của công ty.*", r"vào ngày.*",
        r"đến ngày.*", r"tại ngày.*", r"cuối năm.*", r"đầu năm.*",
        r"trong năm.*", r"năm.*", r"báo cáo tài chính.*", r"báo cáo riêng.*", r"báo cáo hợp nhất.*",
    ]
    for p in patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE)

    prefix_patterns = [
        r"^\s*tổng\s+số\s+", r"^\s*tổng\s+", r"^\s*số\s+dư\s+",
        r"^\s*giá\s+trị\s+", r"^\s*chỉ\s+tiêu\s+",
    ]
    for pp in prefix_patterns:
        text = re.sub(pp, "", text, flags=re.IGNORECASE)

    cleaned = text.strip(" ,.?:;\t\n")
    return cleaned if len(cleaned) >= 2 else noi_dung_input.strip()

def data_discovery_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()
    user_query = state.get("user_query", "")
    parsed_query = state.get("parsed_query", {})
    ten_cong_ty = parsed_query.get("ten_cong_ty", "")
    so_nam = parsed_query.get("so_nam", [])
    noi_dung_raw = parsed_query.get("noi_dung", "")
    thao_tac = parsed_query.get("thao_tac") or parsed_query.get("muc_tieu", "trich_xuat")

    if not so_nam and isinstance(user_query, str):
        so_nam = re.findall(r"\b(20\d{2})\b", user_query)
    if isinstance(user_query, str) and not ten_cong_ty:
        m_ticker = re.search(r"\b([A-Za-z]{3,5})\b", user_query)
        if m_ticker:
            ten_cong_ty = m_ticker.group(1).upper()

    if not noi_dung_raw:
        noi_dung_raw = user_query if isinstance(user_query, str) else ""

    noi_dung = clean_query_content(noi_dung_raw, ten_cong_ty, so_nam)

    print(f"\n🔍 [Data Discovery] Bắt đầu tìm kiếm dữ liệu...")
    print(f"   - Công ty: '{ten_cong_ty}', Số năm: {so_nam}, Nội dung đã làm sạch: '{noi_dung}' (gốc: '{noi_dung_raw}')")

    report_type = None
    if isinstance(user_query, str):
        q_lower = user_query.lower()
        if "hợp nhất" in q_lower and "riêng" not in q_lower:
            report_type = "consolidated"
        elif "báo cáo riêng" in q_lower:
            report_type = "separate"

    all_discovered_tables: List[Dict[str, Any]] = []

    try:
        from rag_module.search_engine import search_by_company_and_content
        import rag_module.search_engine as se
        se._ensure_resources()

        if not so_nam:
            results = search_by_company_and_content(
                company_name=ten_cong_ty, content=noi_dung, year=None, report_type=report_type, top_k=5
            )
            if not results and report_type is not None:
                results = search_by_company_and_content(
                    company_name=ten_cong_ty, content=noi_dung, year=None, report_type=None, top_k=5
                )
            if results:
                _log_candidates(results)
                for match in results[:3]:
                    csv_path = _resolve_csv_path(match.get("csv_path", ""), cfg)
                    if csv_path:
                        table_entry = {
                            "csv_path": str(csv_path),
                            "Ten_Bang": match.get("Ten_Bang", ""),
                            "rrf_score": match.get("rrf_score", 0.0),
                            "Ma_Doanh_Nghiep": match.get("Ma_Doanh_Nghiep", ten_cong_ty),
                            "Nam_Tai_Chinh": match.get("Nam_Tai_Chinh", ""),
                            "Loai_Bao_Cao": match.get("Loai_Bao_Cao", ""),
                                "matched_sample": match.get("matched_sample", ""),
                        }
                        if not any(t["csv_path"] == str(csv_path) for t in all_discovered_tables):
                            all_discovered_tables.append(table_entry)
        else:
            for year in so_nam:
                results = search_by_company_and_content(
                    company_name=ten_cong_ty, content=noi_dung, year=str(year), report_type=report_type, top_k=5
                )
                if not results and report_type is not None:
                    results = search_by_company_and_content(
                        company_name=ten_cong_ty, content=noi_dung, year=str(year), report_type=None, top_k=5
                    )
                if results:
                    _log_candidates(results, year_label=str(year))
                    for match in results[:3]:
                        csv_path = _resolve_csv_path(match.get("csv_path", ""), cfg)
                        if csv_path:
                            table_entry = {
                                "csv_path": str(csv_path),
                                "Ten_Bang": match.get("Ten_Bang", ""),
                                "rrf_score": match.get("rrf_score", 0.0),
                                "Ma_Doanh_Nghiep": match.get("Ma_Doanh_Nghiep", ten_cong_ty),
                                "Nam_Tai_Chinh": str(year),
                                "Loai_Bao_Cao": match.get("Loai_Bao_Cao", ""),
                                "matched_sample": match.get("matched_sample", ""),
                            }
                            if not any(t["csv_path"] == str(csv_path) for t in all_discovered_tables):
                                all_discovered_tables.append(table_entry)
    except Exception as e:
        print(f"⚠️ [Data Discovery] Lỗi Search Engine: {e}")

    latency = time.time() - start_time
    node_latencies = state.get("node_latencies", {})
    node_latencies["data_discovery"] = round(latency, 3)

    if not all_discovered_tables:
        return {
            **state,
            "status": "error",
            "error_message": "Không tìm thấy bảng dữ liệu phù hợp.",
            "discovered_tables": [],
            "node_latencies": node_latencies,
        }

    return {
        **state,
        "discovered_tables": all_discovered_tables,
        "matched_table_path": all_discovered_tables[0]["csv_path"],
        "status": "pending",
        "node_latencies": node_latencies,
    }


### Node 3: Schema Mapper Node
Ánh xạ `tieu_chi_phu` sang tên cột thực tế trong bảng CSV (rule-based fuzzy matching).

In [ ]:
import pandas as pd
from thefuzz import process, fuzz

DEFAULT_VALUE_COLUMNS = [
    "Năm nay", "Năm trước",
    "Số cuối năm", "Số đầu năm",
    "Số cuối kỳ", "Số đầu kỳ",
    "Kỳ này", "Kỳ trước",
]

METADATA_HEADER_COLUMNS = [
    "Ma_Doanh_Nghiep", "Ten_Doanh_Nghiep", "Nam_Tai_Chinh",
    "Loai_Bao_Cao", "Ten_Bang", "Don_Vi_Tinh", "Tep_Nguon"
]

KNOWN_LABEL_COLUMNS = [
    "CHÍ TIÊU", "CHỈ TIÊU", "TÀI SẢN", "NGUỒN VỐN",
    "Cột_0", "Chỉ tiêu", "Mã số", "STT"
]

def _get_columns_from_table(table: Dict[str, Any]) -> List[str]:
    csv_path = table.get("csv_path", "")
    if not csv_path:
        return []
    try:
        df = pd.read_csv(csv_path, nrows=2)
        return list(df.columns)
    except Exception:
        return []

def _find_label_column(columns: List[str]) -> Optional[str]:
    for c in KNOWN_LABEL_COLUMNS:
        if c in columns:
            return c
    for c in columns:
        if c not in METADATA_HEADER_COLUMNS:
            return c
    return columns[0] if columns else None

def _find_value_column(columns: List[str], label_col: Optional[str] = None, tieu_chi_phu: Optional[str] = None) -> Optional[str]:
    label_idx = columns.index(label_col) if label_col and label_col in columns else -1
    value_candidate_cols = []
    for idx, c in enumerate(columns):
        if c in METADATA_HEADER_COLUMNS or c == label_col:
            continue
        if idx > label_idx or label_idx == -1:
            value_candidate_cols.append(c)

    if tieu_chi_phu and value_candidate_cols:
        clean_tcp = str(tieu_chi_phu).strip().lower()
        for col in value_candidate_cols:
            if clean_tcp in col.strip().lower():
                return col
        match, score = process.extractOne(
            tieu_chi_phu, value_candidate_cols, scorer=fuzz.token_set_ratio
        )
        if score >= 50:
            return match

    for c in DEFAULT_VALUE_COLUMNS:
        if c in value_candidate_cols:
            return c

    if value_candidate_cols:
        return value_candidate_cols[0]
    return None

def schema_mapper_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()
    parsed_query = state.get("parsed_query", {})
    discovered_tables = state.get("discovered_tables", [])
    tieu_chi_phu = parsed_query.get("tieu_chi_phu")

    print(f"\n🔍 [Schema Mapper] Đang ánh xạ tiêu chí → cột thực tế...")
    column_mapping: Dict[str, str] = {}

    if not discovered_tables:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["schema_mapper"] = round(latency, 3)
        return {**state, "column_mapping": {}, "status": "pending", "node_latencies": node_latencies}

    first_table = discovered_tables[0]
    columns = _get_columns_from_table(first_table)
    if not columns:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["schema_mapper"] = round(latency, 3)
        return {**state, "column_mapping": {}, "status": "pending", "node_latencies": node_latencies}

    label_col = _find_label_column(columns)
    if label_col:
        column_mapping["label_column"] = label_col
    value_col = _find_value_column(columns, label_col, tieu_chi_phu)
    if value_col:
        column_mapping["value_column"] = value_col
    column_mapping["all_columns"] = str(columns)

    print(f"📊 [Kết quả - Schema Mapper]: {column_mapping}\n")

    latency = time.time() - start_time
    node_latencies = state.get("node_latencies", {})
    node_latencies["schema_mapper"] = round(latency, 3)

    return {**state, "column_mapping": column_mapping, "status": "pending", "node_latencies": node_latencies}

print("✅ Schema Mapper node loaded!")


### Node 4: Code Generator & Reflection Node
Sinh mã Pandas xử lý câu hỏi tài chính và hỗ trợ Reflection Debugging Loop khi xảy ra lỗi.

In [ ]:
def code_generator_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()

    user_query = state.get("user_query", "")
    parsed_query = state.get("parsed_query", {})
    discovered_tables = state.get("discovered_tables", [])
    column_mapping = state.get("column_mapping", {})
    if not column_mapping and discovered_tables:
        try:
            temp_state = schema_mapper_node(state, cfg)
            column_mapping = temp_state.get("column_mapping", {})
            state["column_mapping"] = column_mapping
        except Exception as e:
            print(f"⚠️ Inline schema mapping failed: {e}")
    error_traceback = state.get("error_traceback")
    retry_count = state.get("retry_count", 0)

    muc_tieu = parsed_query.get("muc_tieu", "trich_xuat")
    noi_dung = parsed_query.get("noi_dung", "")
    ten_cong_ty = parsed_query.get("ten_cong_ty", "")
    so_nam = parsed_query.get("so_nam", [])
    tieu_chi_phu = parsed_query.get("tieu_chi_phu")

    label_col = column_mapping.get("label_column", "CHỈ TIÊU")
    value_col = column_mapping.get("value_column", "Năm nay")

    files_context = _build_files_context(discovered_tables, column_mapping)

    paths_str = ""
    if discovered_tables:
        if len(discovered_tables) == 1:
            escaped = discovered_tables[0]["csv_path"].replace('\\', '/')
            paths_str = f"file_path = '{escaped}'"
        else:
            for tbl in discovered_tables:
                nam = tbl.get("Nam_Tai_Chinh", "default")
                escaped = tbl["csv_path"].replace('\\', '/')
                paths_str += f"file_path_{nam} = '{escaped}'\n"

    try:
        if not error_traceback or retry_count == 0:
            prompt_data = PROMPT_CODE_GENERATOR
            system_prompt = prompt_data.get("system_prompt", "")
            few_shots = prompt_data.get("few_shot_examples", [])
            goal_descs = prompt_data.get("goal_descriptions", {})
            goal_instructions = prompt_data.get("goal_instructions", {})

            messages = [SystemMessage(content=system_prompt)]
            for ex in few_shots:
                messages.append(HumanMessage(content=f"Yêu cầu: {ex['user_query']}\nFile Path: {ex['file_path']}\nColumn Mapping: {ex['column_mapping']}"))
                messages.append(SystemMessage(content=ex["generated_code"]))

            goal_desc = goal_descs.get(muc_tieu, muc_tieu)
            goal_inst_template = goal_instructions.get(muc_tieu, "")
            goal_inst = goal_inst_template.format(
                noi_dung=noi_dung, label_col=label_col, value_col=value_col
            ) if goal_inst_template else ""

            user_template = prompt_data.get("user_prompt_template", "")
            human_content = user_template.format(
                user_query=user_query,
                muc_tieu_desc=goal_desc,
                noi_dung=noi_dung,
                ten_cong_ty=ten_cong_ty,
                so_nam=so_nam,
                tieu_chi_phu=tieu_chi_phu or "(không có)",
                files_context=files_context,
                label_col=label_col,
                value_col=value_col,
                paths_str=paths_str,
                goal_instruction=goal_inst,
            )
            messages.append(HumanMessage(content=human_content))
        else:
            prompt_data = PROMPT_REFLECTION
            system_prompt = prompt_data.get("system_prompt", "")

            print(f"🔄 [Reflection Loop] Đang sửa lỗi mã nguồn (Lần {retry_count})...")

            retry_forcing_msg = (
                f"Execution failed with error: {error_traceback.strip()}\n"
                "CRITICAL: The string you used in `str.contains()` was NOT found in the table. "
                "DO NOT output the exact same code again! "
                "You MUST change your search strategy: shorten the search string in `str.contains(..., regex=False)` to a single core keyword from the metric, or inspect the sample row labels below."
            )

            sample_labels = []
            if discovered_tables:
                for tbl in discovered_tables:
                    c_path = tbl.get("csv_path")
                    if c_path and Path(c_path).exists():
                        try:
                            sub_df = pd.read_csv(c_path)
                            if label_col in sub_df.columns:
                                labels = sub_df[label_col].dropna().astype(str).head(20).tolist()
                                sample_labels.append(f"Mẫu chỉ tiêu thực tế trong file '{Path(c_path).name}':\n{labels}")
                        except Exception:
                            pass
            sample_labels_str = "\n\n".join(sample_labels) if sample_labels else ""

            user_template = prompt_data.get("user_prompt_template", "")
            human_content = user_template.format(
                user_query=user_query,
                muc_tieu=muc_tieu,
                noi_dung=noi_dung,
                files_context=files_context,
                label_col=label_col,
                value_col=value_col,
                paths_str=paths_str,
                sample_labels_str=sample_labels_str,
                previous_code=state.get('generated_code', ''),
                error_traceback=f"{error_traceback.strip()}\n\n{retry_forcing_msg}",
            )
            messages = [SystemMessage(content=system_prompt), HumanMessage(content=human_content)]

        llm = get_llm(cfg=cfg, temperature=0.0)
        response = llm.invoke(messages)
        raw_text = response.content if isinstance(response.content, str) else str(response.content)

        code = clean_python_code(raw_text)
        print(f"📊 [Kết quả - Code Generator] Mã Python sinh ra:\n```python\n{code}\n```\n")

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["code_generator"] = round(latency, 3)

        return {**state, "generated_code": code, "status": "pending", "node_latencies": node_latencies}

    except Exception as e:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["code_generator"] = round(latency, 3)
        return {**state, "status": "error", "error_message": f"Code generator node error: {str(e)}", "node_latencies": node_latencies}


### Node 5: AST Sandbox & Executor Node
Thực thi mã Python trong môi trường Sandbox AST an toàn và thu thập kết quả `result`.

In [ ]:
import ast
import sys
import time
import traceback
import pandas as pd
import numpy as np
from typing import Dict, Any, Optional

class SecurityError(Exception):
    """Raised when generated code contains forbidden AST nodes."""
    pass

FORBIDDEN_AST_NODES = (ast.Import, ast.ImportFrom)
FORBIDDEN_BUILTINS = {
    "eval", "exec", "__import__", "open", "compile",
    "globals", "locals", "input", "breakpoint"
}
ALLOWED_MODULES = {"pandas", "pd", "numpy", "np", "datetime", "math", "re"}

def validate_ast(code_str: str) -> None:
    tree = ast.parse(code_str)
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in ALLOWED_MODULES:
                    raise SecurityError(f"Importing forbidden module: '{alias.name}'")
        elif isinstance(node, ast.ImportFrom):
            if node.module and node.module.split(".")[0] not in ALLOWED_MODULES:
                raise SecurityError(f"Importing from forbidden module: '{node.module}'")
        elif isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id in FORBIDDEN_BUILTINS:
                raise SecurityError(f"Call to forbidden function: '{node.func.id}'")

def format_result(result: Any) -> Any:
    if isinstance(result, pd.DataFrame):
        df_sub = result.head(100)
        return {
            "type": "dataframe",
            "shape": list(result.shape),
            "columns": list(result.columns),
            "data": df_sub.to_dict(orient="records"),
        }
    elif isinstance(result, pd.Series):
        s_sub = result.head(100)
        return {
            "type": "series",
            "name": str(result.name) if result.name else "result",
            "data": s_sub.to_dict(),
        }
    elif isinstance(result, (int, float, str, bool, list, dict)):
        return {
            "type": "scalar",
            "data": result,
        }
    else:
        return {
            "type": "other",
            "data": str(result),
        }

def executor_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()

    code_str = state.get("generated_code", "").strip()
    discovered_tables = state.get("discovered_tables", [])
    file_path = ""
    if discovered_tables:
        file_path = discovered_tables[0].get("csv_path", "")
    retry_count = state.get("retry_count", 0)

    if not code_str:
        return {
            **state,
            "status": "error",
            "error_traceback": "No code generated to execute.",
            "retry_count": retry_count + 1,
        }

    try:
        validate_ast(code_str)
        print(f"⚙️ [Executor] Đang thực thi mã Pandas...")

        df_loaded = None
        if file_path and os.path.exists(file_path):
            try:
                df_loaded = pd.read_csv(file_path)
            except Exception:
                pass

        # Robust clean_val definition to inject as a fail-safe
        def clean_val(val):
            if pd.isna(val) or str(val).strip() in ['-', '', 'nan', 'NaN', 'None', 'null', 'n/a', '—']:
                raise ValueError("Metric not found in table")
            if isinstance(val, (int, float)): return float(val)
            val = str(val).strip()
            neg = False
            if val.startswith('(') and val.endswith(')'):
                neg = True
                val = val[1:-1].strip()
            val = val.replace(',', '')
            if '.' in val:
                parts = val.split('.')
                if len(parts) > 2 or (len(parts) == 2 and len(parts[1]) == 3):
                    val = val.replace('.', '')
            try:
                res = float(val)
                return -res if neg else res
            except Exception:
                raise ValueError("Metric not found in table")

        exec_globals = {
            "pd": pd,
            "np": np,
            "pandas": pd,
            "numpy": np,
            "file_path": file_path,
            "df": df_loaded,
            "clean_val": clean_val,
        }
        for tbl in discovered_tables:
            csv_p = tbl.get("csv_path", "")
            nam = tbl.get("Nam_Tai_Chinh", "")
            if csv_p and nam:
                exec_globals[f"file_path_{nam}"] = csv_p

        exec(code_str, exec_globals)

        result_val = exec_globals.get("result")
        if result_val is None:
            raise ValueError("Biến `result` không được tìm thấy sau khi thực thi mã.")

        formatted = format_result(result_val)
        print(f"✅ [Executor] Thực thi THÀNH CÔNG!")
        print(f"📊 [Kết quả - Executor]:\n{json.dumps(formatted, indent=4, ensure_ascii=False)}\n")

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["executor"] = round(latency, 3)

        return {
            **state,
            "execution_result": formatted,
            "error_traceback": None,
            "status": "success",
            "node_latencies": node_latencies,
        }

    except Exception as e:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["executor"] = round(latency, 3)
        tb_str = traceback.format_exc()
        return {
            **state,
            "status": "error",
            "error_traceback": tb_str,
            "retry_count": retry_count + 1,
            "node_latencies": node_latencies,
        }

print("✅ Executor node loaded!")


## 🌐 Section 5: LangGraph Workflow & Edge Routing
Xây dựng đồ thị trạng thái Agent StateGraph và thiết lập điều kiện Reflection Loop.

In [ ]:
from langgraph.graph import StateGraph, END

def route_after_discovery(state: AgentState) -> Literal["code_generator", "__end__"]:
    if state.get("status") == "error":
        return END
    return "code_generator"

def route_after_execution(state: AgentState, cfg: Optional[Config] = None) -> Literal["code_generator", "__end__"]:
    cfg = cfg or config
    status = state.get("status")
    retry_count = state.get("retry_count", 0)
    if status == "success":
        return END
    if status == "error" and retry_count < cfg.MAX_RETRIES:
        print(f"🔄 Reflection Loop Activated! Retrying code generation ({retry_count}/{cfg.MAX_RETRIES})...")
        return "code_generator"
    return END

def create_cocopila_graph(cfg: Optional[Config] = None):
    cfg = cfg or config
    workflow = StateGraph(AgentState)
    # Add Nodes
    workflow.add_node("query_parser", lambda s: query_parser_node(s, cfg))
    workflow.add_node("data_discovery", lambda s: data_discovery_node(s, cfg))
    workflow.add_node("code_generator", lambda s: code_generator_node(s, cfg))
    workflow.add_node("executor", lambda s: executor_node(s, cfg))
    # Add Edges
    workflow.set_entry_point("query_parser")
    workflow.add_edge("query_parser", "data_discovery")
    workflow.add_conditional_edges("data_discovery", route_after_discovery, {"code_generator": "code_generator", END: END})
    workflow.add_edge("code_generator", "executor")
    workflow.add_conditional_edges("executor", lambda s: route_after_execution(s, cfg), {"code_generator": "code_generator", END: END})
    return workflow.compile()

print("✅ Đã khởi tạo thành công hàm create_cocopila_graph()!")


## 🧪 Section 6: Dataset Linking & Running Agent Test

In [ ]:
# 2. Dò tìm và liên kết trực tiếp Qdrant Local DB từ đường dẫn chỉ định trên Kaggle
import os
import shutil
from pathlib import Path

# Các đường dẫn khả thi trên Kaggle (bao gồm cả URL trình duyệt và đường dẫn hệ thống thực tế)
possible_dataset_dirs = [
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data"),
    Path("/kaggle/input/r2-ai-output/r2AI_data"),
    Path("/kaggle/input/r2-ai-output"),
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output"),
]

if Path("/kaggle/input").exists():
    print("🔍 Đang kiểm tra Kaggle Input Dataset cho Qdrant DB...")
    dataset_dir = None
    
    # 1. Thử các đường dẫn chỉ định trước
    for d in possible_dataset_dirs:
        if (d / "qdrant_local_db").exists():
            dataset_dir = d
            print(f"📦 Đã tìm thấy Qdrant DB tại đường dẫn chỉ định: {d / 'qdrant_local_db'}")
            break
            
    # 2. Nếu không thấy, dùng quét nông (shallow search) để tìm kiếm tự động
    if not dataset_dir:
        qdrant_found = []
        for depth_pattern in [
            "*/qdrant_local_db", 
            "*/*/qdrant_local_db", 
            "*/*/*/qdrant_local_db", 
            "*/*/*/*/qdrant_local_db", 
            "*/*/*/*/*/qdrant_local_db"
        ]:
            qdrant_found.extend(list(Path("/kaggle/input").glob(depth_pattern)))
            if qdrant_found:
                dataset_dir = qdrant_found[0].parent
                print(f"📦 Đã tìm thấy Qdrant DB bằng wildcard tại: {qdrant_found[0]}")
                break
            
    if dataset_dir:
        rag_module_dir = Path("rag_module").resolve()
        rag_module_dir.mkdir(exist_ok=True)
        
        # Symlink cho các file chỉ đọc (ViFinQA, bm25, code_stock)
        for item in ["bm25_index.pkl", "code_stock.csv", "ViFinQA"]:
            src = dataset_dir / item
            dst = rag_module_dir / item
            
            # Xóa liên kết cũ/hỏng nếu có để tránh lỗi FileExistsError
            if dst.exists() or dst.is_symlink():
                try:
                    if dst.is_symlink() or dst.is_file():
                        dst.unlink()
                    else:
                        shutil.rmtree(dst)
                except Exception as e:
                    print(f"   ⚠️ Cannot remove old {item}: {e}")
                    
            if src.exists() and not dst.exists():
                try:
                    os.symlink(src, dst)
                    print(f"   🔗 Created symlink: {dst} -> {src}")
                except Exception as e:
                    print(f"   ⚠️ Cannot symlink {item}: {e}")
        
        # Copy vật lý cho qdrant_local_db vì Qdrant yêu cầu quyền ghi (.lock file)
        qdrant_src = dataset_dir / "qdrant_local_db"
        qdrant_dst = rag_module_dir / "qdrant_local_db"
        
        # Xóa liên kết cũ/hỏng nếu có để tránh lỗi FileExistsError khi copytree
        if qdrant_dst.exists() or qdrant_dst.is_symlink():
            try:
                if qdrant_dst.is_symlink() or qdrant_dst.is_file():
                    qdrant_dst.unlink()
                else:
                    shutil.rmtree(qdrant_dst)
            except Exception as e:
                print(f"   ⚠️ Cannot remove old qdrant_local_db: {e}")
                
        if qdrant_src.exists() and not qdrant_dst.exists():
            print("   ⏳ Đang copy Qdrant DB sang thư mục làm việc để cấp quyền ghi (chỉ mất vài chục giây cho lần đầu)...")
            shutil.copytree(qdrant_src, qdrant_dst)
            print(f"   ✅ Đã copy xong Qdrant DB tới: {qdrant_dst}")
    else:
        print("❌ Không tìm thấy thư mục dataset trên Kaggle! Hãy kiểm tra xem bạn đã đính kèm dataset vào Notebook chưa.")
else:
    print("💻 Chạy local, sử dụng dữ liệu có sẵn tại rag_module/")

In [ ]:
# 3. Khởi tạo Agent và thực thi 10 câu hỏi ngẫu nhiên (BOUNDED REFLECTION & ERROR HANDLING)
import time
import random
import json
from pathlib import Path
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

repo_dir = Path("/kaggle/working/r2AI_2026")
if not repo_dir.exists():
    repo_dir = Path.cwd()

# Nạp danh sách câu hỏi kiểm thử từ dataset
possible_qa_paths = [
    repo_dir / "rag_module" / "ViFinQA" / "questions" / "questions.jsonl",
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output/rag_module/ViFinQA/questions/questions.jsonl"),
    Path("/kaggle/input/r2-ai-output/rag_module/ViFinQA/questions/questions.jsonl"),
    repo_dir / "rag_module" / "ViFinQA" / "ViFinQA_QA.json",
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data/ViFinQA/ViFinQA_QA.json"),
    Path("rag_module/ViFinQA/ViFinQA_QA.json"),
]

qa_file = None
for p in possible_qa_paths:
    if p.exists():
        qa_file = p
        break

qa_data = []
if qa_file and qa_file.exists():
    with open(qa_file, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if qa_file.suffix == ".jsonl" or "\n" in content:
            for line in content.splitlines():
                if line.strip():
                    try:
                        qa_data.append(json.loads(line))
                    except Exception:
                        pass
        else:
            try:
                qa_data = json.loads(content)
            except Exception:
                pass
    print(f"📋 Đã tải thành công {len(qa_data)} câu hỏi từ dataset!")
else:
    print("⚠️ Không tìm thấy file ViFinQA_QA.json, sử dụng câu hỏi mẫu mặc định.")
    qa_data = [
        {"id": 655, "question": "Tốc độ tăng trưởng % tổng tiền và các khoản tương đương tiền của VIC từ năm 2019 đến năm 2021 là bao nhiêu?"},
        {"id": 115, "question": "Tổng chi phí hoạt động của công ty mẹ Ngân hàng TMCP Công Thương Việt Nam năm 2019 là bao nhiêu triệu đồng?"},
        {"id": 26, "question": "Lãi vay phải trả của CTCP Tập đoàn Đức Long Gia Lai (DLG) cuối năm 2023 là bao nhiêu triệu đồng?"},
    ]

random.seed(42)
sample_questions = qa_data[:10] if len(qa_data) >= 10 else qa_data

results_summary = []
MAX_ATTEMPTS = 3

for idx, q_item in enumerate(sample_questions, 1):
    q_id = q_item.get("id", idx)
    q_text = q_item.get("question", "")
    print(f"\n[{idx}/{len(sample_questions)}] ❓ Câu hỏi ID {q_id}: {q_text}")
    print("-" * 60)
    
    try:
        # 1. STRICT STATELESS PARSER (Fixing Memory Leak)
        parser_prompt_data = PROMPT_QUERY_PARSER
        sys_prompt = parser_prompt_data.get("system_prompt", "")
        few_shots = parser_prompt_data.get("few_shot_examples", [])
        
        parser_messages = [SystemMessage(content=sys_prompt)]
        for ex in few_shots:
            parser_messages.append(HumanMessage(content=ex["user_query"]))
            parser_messages.append(SystemMessage(content=ex["parsed_output"]))
        parser_messages.append(HumanMessage(content=f"Câu hỏi: {q_text}"))
        
        llm_parser = get_llm(config, temperature=0.1, timeout=30)
        try:
            response_parser = llm_parser.invoke(parser_messages)
            raw_parser_out = response_parser.content if hasattr(response_parser, "content") else str(response_parser)
        except Exception as api_err:
            print(f"   ⏱️ LLM Parser API call failed or timed out: {api_err}")
            results_summary.append({"id": q_id, "question": q_text, "status": "error", "error": "Parser timeout"})
            continue

        parsed_json = safe_parse_json(raw_parser_out)
        
        ticker_val = parsed_json.get("ticker") or parsed_json.get("ten_cong_ty") or ""
        metric_val = parsed_json.get("metric") or parsed_json.get("noi_dung") or ""
        year_val = parsed_json.get("year") if "year" in parsed_json else parsed_json.get("so_nam")
        if not year_val or year_val is None or str(year_val).strip() in ["None", "null", ""]:
            year_val = re.findall(r"\b(20\d{2})\b", q_text)
        
        if isinstance(year_val, str):
            so_nam_list = [y.strip() for y in year_val.replace(",", " ").split() if y.strip().isdigit()]
            if not so_nam_list:
                so_nam_list = re.findall(r"\b(20\d{2})\b", q_text)
        elif isinstance(year_val, (int, float)):
            so_nam_list = [str(int(year_val))]
        elif isinstance(year_val, list):
            so_nam_list = [str(y).strip() for y in year_val if str(y).strip().isdigit()]
        else:
            so_nam_list = re.findall(r"\b(20\d{2})\b", q_text)
            
        company_clean = _normalize_company_name(ticker_val, q_text)
        parsed_json["ticker"] = ticker_val
        parsed_json["ten_cong_ty"] = company_clean
        parsed_json["year"] = ", ".join(so_nam_list) if so_nam_list else ""
        parsed_json["so_nam"] = so_nam_list
        parsed_json["metric"] = metric_val
        parsed_json["noi_dung"] = metric_val
        
        thao_tac = "so_sanh" if len(so_nam_list) > 1 or "so sánh" in q_text.lower() or "tăng trưởng" in q_text.lower() else "trich_xuat"
        parsed_json["thao_tac"] = thao_tac
        parsed_json["muc_tieu"] = thao_tac
        
        print(f"📊 [Query Parser Result]: Ticker={company_clean}, Year={so_nam_list}, Metric='{metric_val}'")
        
        state = {
            "user_query": q_text,
            "parsed_query": parsed_json,
            "discovered_tables": [],
            "column_mapping": {},
            "generated_code": "",
            "execution_result": None,
            "error_traceback": None,
            "retry_count": 0,
            "status": "pending",
            "error_message": None,
            "node_latencies": {},
        }
        
        state = data_discovery_node(state, config)
        if state.get("status") == "error":
            print(f"   ❌ Data Discovery Failed: {state.get('error_message')}")
            results_summary.append({"id": q_id, "question": q_text, "status": "error", "error": state.get('error_message')})
            continue
            
        state = schema_mapper_node(state, config)
        
        # 2. CODE GEN & REFLECTION LOOP (Bounded Attempts & Standardized Exceptions)
        discovered_tables = state.get("discovered_tables", [])
        column_mapping = state.get("column_mapping", {})
        label_col = column_mapping.get("label_column", "CHỈ TIÊU")
        value_col = column_mapping.get("value_column", "Năm nay")
        files_context = _build_files_context(discovered_tables, column_mapping)
        
        paths_str = ""
        if discovered_tables:
            if len(discovered_tables) == 1:
                escaped = discovered_tables[0]["csv_path"].replace('\\', '/')
                paths_str = f"file_path = '{escaped}'"
            else:
                for tbl in discovered_tables:
                    nam = tbl.get("Nam_Tai_Chinh", "default")
                    escaped = tbl["csv_path"].replace('\\', '/')
                    paths_str += f"file_path_{nam} = '{escaped}'\n"
        
        cg_prompt_data = PROMPT_CODE_GENERATOR
        cg_sys_prompt = cg_prompt_data.get("system_prompt", "")
        goal_desc = cg_prompt_data.get("goal_descriptions", {}).get(thao_tac, thao_tac)
        goal_inst_template = cg_prompt_data.get("goal_instructions", {}).get(thao_tac, "")
        goal_inst = goal_inst_template.format(noi_dung=metric_val, label_col=label_col, value_col=value_col) if goal_inst_template else ""
        
        user_template = cg_prompt_data.get("user_prompt_template", "")
        human_init_content = user_template.format(
            user_query=q_text,
            muc_tieu_desc=goal_desc,
            noi_dung=metric_val,
            ten_cong_ty=company_clean,
            so_nam=so_nam_list,
            tieu_chi_phu="(không có)",
            files_context=files_context,
            label_col=label_col,
            value_col=value_col,
            paths_str=paths_str,
            goal_instruction=goal_inst,
        )
        
        code_messages = [
            SystemMessage(content=cg_sys_prompt),
            HumanMessage(content=human_init_content)
        ]
        
        llm_codegen = get_llm(config, temperature=0.0, timeout=30)
        execution_success = False
        
        for attempt in range(MAX_ATTEMPTS):
            print(f"⚙️ [Code Generator] Generation attempt {attempt + 1}/{MAX_ATTEMPTS}...")
            try:
                response_code = llm_codegen.invoke(code_messages)
                raw_code = response_code.content if hasattr(response_code, "content") else str(response_code)
            except Exception as api_err:
                print(f"   ⏱️ LLM CodeGen API call failed or timed out on attempt {attempt + 1}: {api_err}")
                break
                
            code = clean_python_code(raw_code)
            state["generated_code"] = code
            
            # Execute code inside AST Sandbox safely (catches all exceptions including TypeError and ValueError)
            try:
                exec_output = executor_node(state, config)
            except Exception as exec_err:
                exec_output = {"status": "error", "error_traceback": f"Execution error: {str(exec_err)}"}
                
            if exec_output.get("status") == "success":
                print(f"✅ [Code Generator] Execution SUCCESS on attempt {attempt + 1}!")
                execution_success = True
                results_summary.append({"id": q_id, "question": q_text, "status": "success", "result": exec_output.get("execution_result")})
                break
            else:
                err_msg = exec_output.get("error_traceback", "Execution failed")
                last_err_line = err_msg.splitlines()[-1] if err_msg else "Execution failed"
                print(f"⚠️ Attempt {attempt + 1}/{MAX_ATTEMPTS} failed: {last_err_line}")
                
                # Check if this was the last attempt: DO NOT append new messages or continue retry loop
                if attempt >= MAX_ATTEMPTS - 1:
                    print(f"🛑 Reached maximum attempts ({MAX_ATTEMPTS}). Gracefully terminating reflection loop for Question ID {q_id}.")
                    break
                
                # IMPROVED REFLECTION PROMPT
                error_instruction = (
                    f"Execution Failed: {last_err_line}."
                    "If your previous attempt failed because the row was not found, DO NOT write the exact same string matching code again. "
                    "You must try matching a different variation of the row name, or use partial string matching (str.contains)."
                    "CRITICAL: Output ONLY executable Python code. NO markdown formatting. NO ```python tags. NO conversational text or explanations. Start directly with 'import pandas'."
                )
                
                # Appends previous assistant response and retry human prompt safely
                code_messages.append(AIMessage(content=f"```python\n{code}\n```"))
                code_messages.append(HumanMessage(content=error_instruction))
        
        if not execution_success:
            print(f"❌ Question ID {q_id} failed after reflection/retry attempts.")
            results_summary.append({"id": q_id, "question": q_text, "status": "error", "error": "Failed after reflection attempts", "result": {"type": "error", "data": None}})

    except Exception as e:
        print(f"   💥 Lỗi ngoại lệ trong quá trình chạy: {e}")
        results_summary.append({"id": q_id, "question": q_text, "status": "error", "error": str(e), "result": {"type": "error", "data": None}})

print("\n" + "=" * 80)
print("📊 TỔNG HỢP KẾT QUẢ KIỂM THỬ:")
print("=" * 80)
success_count = sum(1 for r in results_summary if r["status"] == "success")
print(f"Tổng số câu hỏi: {len(results_summary)} | Thành công: {success_count} | Thất bại: {len(results_summary) - success_count}")
for r in results_summary:
    status_emoji = "✅" if r["status"] == "success" else "❌"
    print(f"{status_emoji} ID {r['id']}: {r['status'].upper()}")
